# Regresión Lineal paso a paso con scikit-learn 

---

> **Pregunta orientadora:**
> ¿Existe una relación lineal entre la temperatura de referencia y la temperatura medida?
> Y si existe, ¿cuál es esa relación (ecuación de la recta) y qué tan buena es?

### Plan del notebook:

1. Importar las herramientas (librerías) que vamos a usar.
2. Cargar los datos del experimento en una tabla (DataFrame).
3. Agregar un **factor de ruido controlable** para simular errores de medición.
4. Preparar los datos para scikit-learn.
5. Entrenar (ajustar) el modelo de Regresión Lineal.
6. Calcular las predicciones del modelo.
7. Calcular el **error residual**, el **RMSE** y el **R²**.
8. Graficar los puntos originales junto con la recta de regresión.

---

## Paso 1 — Importar las librerías

In [ ]:
# pandas: nos permite trabajar con tablas de datos (como una hoja de Excel), llamadas DataFrames.
import pandas as pd

# numpy: nos permite trabajar con números y arreglos (arrays) de forma eficiente.
# Lo usaremos, por ejemplo, para generar el "ruido" aleatorio.
import numpy as np

# matplotlib: es la librería que usamos para hacer gráficos (dibujar puntos, líneas, etc.)
import matplotlib.pyplot as plt

# De scikit-learn (la librería de Machine Learning) importamos:
# - LinearRegression: el modelo que va a "aprender" la relación lineal entre nuestras variables.
from sklearn.linear_model import LinearRegression

# - mean_squared_error y r2_score: funciones que nos ayudan a medir qué tan bueno es el modelo.
from sklearn.metrics import mean_squared_error, r2_score

# Esta línea es solo estética: hace que los gráficos se vean un poco más grandes por defecto.
plt.rcParams["figure.figsize"] = (8, 6)


## Paso 2 — Cargar los datos del experimento

Los datos que tenemos son dos listas de números:

- `t_ref`: la temperatura que marca el termómetro de **referencia** (el que confiamos que es correcto).
- `t_medida`: la temperatura que marca el termómetro que estamos **probando**.

Vamos a guardar estos datos en un **diccionario** de Python y luego convertirlo en una
tabla (DataFrame) para poder verlos y trabajar con ellos más fácilmente.


In [ ]:
# Un diccionario en Python es como una "caja con etiquetas": cada etiqueta (clave)
# apunta a una lista de valores.
datos = {
    "t_ref":    [0, 10, 20, 30, 40, 50, 60, 70],
    "t_medida": [1.5, 8.9, 21.7, 28.4, 42.3, 48.8, 62.0, 69.2]
}

# pd.DataFrame(...) convierte ese diccionario en una tabla (filas y columnas).
df = pd.DataFrame(datos)

print("DATOS DEL EXPERIMENTO")

# display() muestra la tabla de forma "bonita" dentro del notebook (mejor que print).
display(df)

## Paso 3 — Agregar un factor de ruido controlable

En la vida real, **ningún instrumento mide de forma perfecta**: siempre hay un pequeño error aleatorio (esto se llama "ruido"). Para practicar y entender mejor cómo afecta el ruido a una regresión lineal, vamos a crear una variable llamada `nivel_ruido` que controla **cuánto** ruido extra le agregamos a los datos.

- Si `nivel_ruido = 0` → no agregamos ruido, usamos los datos originales.
- Si `nivel_ruido` es un número más grande (por ejemplo `2`, `5`, `10`) → agregamos más "desorden" aleatorio a los datos medidos.

👉 Prueba cambiar el valor de `nivel_ruido` más abajo y vuelve a ejecutar el notebook completo, para ver cómo cambia la calidad de la regresión (el R² y el RMSE).


In [ ]:

# =========================================================
# 🎛️ PANEL DE CONTROL: nivel de ruido
# =========================================================
# Este número representa la "desviación estándar" del ruido que vamos a agregar.
# Cuanto más grande sea, más dispersos (ruidosos) se verán los datos.
# Cambia este valor y vuelve a correr el notebook para experimentar.
nivel_ruido = 20.0
# =========================================================

# Fijamos una "semilla" aleatoria (random seed). Esto hace que, aunque el ruido
# sea "aleatorio", siempre obtengamos el MISMO resultado cada vez que ejecutamos
# el notebook con el mismo nivel_ruido. Esto es muy útil para poder comparar resultados.
np.random.seed(42)

# np.random.normal(media, desviacion_estandar, cantidad_de_valores)
# genera "cantidad_de_valores" números aleatorios que siguen una distribución normal
# (forma de campana), centrados en 0, con una dispersión igual a "nivel_ruido".
ruido = np.random.normal(loc=0, scale=nivel_ruido, size=len(df))

# Creamos una nueva columna en el DataFrame sumando el ruido a la temperatura medida original.
df["t_medida_ruidosa"] = df["t_medida"] + ruido

print(f"Nivel de ruido aplicado: {nivel_ruido}")
print("Tabla con la columna de datos ruidosos agregada:")
display(df)


## Paso 4 — Preparar los datos para scikit-learn

scikit-learn es muy exigente con la "forma" (shape) de los datos:

- La variable **X** (entrada / variable independiente) tiene que ser una tabla de
  **2 dimensiones**, aunque solo tenga una columna. Por eso usamos `.reshape(-1, 1)`.
- La variable **y** (salida / variable dependiente, lo que queremos predecir) puede
  quedarse como una lista simple de 1 dimensión.

En nuestro caso:

- **X** = temperatura de referencia (`t_ref`) → es la variable que "conocemos".
- **y** = temperatura medida (con ruido) (`t_medida_ruidosa`) → es la variable que
  queremos explicar/predecir a partir de X.


In [ ]:

# .values convierte la columna del DataFrame en un array de numpy.
# .reshape(-1, 1) le da la forma de "muchas filas, 1 columna", que es lo que pide scikit-learn.
X = df["t_ref"].values.reshape(-1, 1)

# y se queda como un array de 1 dimensión (una lista de números).
y = df["t_medida_ruidosa"].values

print("Forma (shape) de X:", X.shape)  # (8, 1) -> 8 filas, 1 columna
print("Forma (shape) de y:", y.shape)  # (8,)   -> 8 valores

print("\nAsí se ven los primeros valores de X:")
print(X[:3])

print("\nAsí se ven los primeros valores de y:")
print(y[:3])



## Paso 5 — Crear y entrenar el modelo de Regresión Lineal

La Regresión Lineal busca la **mejor línea recta** que describe la relación entre X e y.
Una línea recta se define con la ecuación:

$$ y = m \cdot X + b $$

Donde:
- **m** (pendiente / "slope"): indica cuánto sube "y" por cada unidad que sube "X".
- **b** (intersección / "intercept"): indica el valor de "y" cuando X = 0.

scikit-learn calcula automáticamente los valores de **m** y **b** que mejor se ajustan
a nuestros datos. Este proceso se llama **"entrenar" o "ajustar" (fit)** el modelo.


In [ ]:

# Creamos una "instancia" (un objeto) del modelo de Regresión Lineal.
# En este momento el modelo está "vacío".
modelo = LinearRegression()

# .fit(X, y) es el paso donde el modelo "aprende":
# encuentra la pendiente (m) y la intersección (b) que mejor ajustan los datos.
modelo.fit(X, y)

# Una vez entrenado, podemos leer los parámetros que el modelo encontró:
pendiente = modelo.coef_[0]      # este es el valor de "m"
interseccion = modelo.intercept_  # este es el valor de "b"

print("MODELO ENTRENADO")
print(f"Pendiente (m):     {pendiente:.4f}")
print(f"Intersección (b):  {interseccion:.4f}")
print(f"\nEcuación de la recta encontrada:")
print(f"   t_medida ≈ {pendiente:.4f} * t_ref + {interseccion:.4f}")


## Paso 6 — Calcular las predicciones del modelo

Ahora que el modelo ya "aprendió" la recta, podemos usarlo para **predecir** qué
temperatura medida esperaríamos para cada temperatura de referencia, usando esa
ecuación de la recta.

Esto se hace con el método `.predict(X)`.


In [ ]:
# .predict(X) le pide al modelo que calcule "y" usando la ecuación de la recta
# que encontró, para cada valor de X que le pasemos.
predicciones = modelo.predict(X)

# Guardamos las predicciones en una nueva columna del DataFrame, para poder
# comparar fácilmente el valor real contra el valor predicho por el modelo.
df["prediccion"] = predicciones

print("Comparación entre el valor real (con ruido) y la predicción del modelo:")
display(df[["t_ref", "t_medida_ruidosa", "prediccion"]])


## Paso 7 — Error residual, RMSE y R²

Ahora vamos a medir **qué tan bien** predice nuestro modelo. Para eso usamos tres
conceptos:

### 1) Error residual (o simplemente "residuo")
Es la diferencia entre el valor **real** y el valor **predicho** por el modelo, para
cada punto:

$$ \text{residuo} = y_{real} - y_{predicho} $$

- Si el residuo es 0 → el modelo predijo exactamente ese punto.
- Si el residuo es positivo → el modelo predijo un valor **menor** que el real.
- Si el residuo es negativo → el modelo predijo un valor **mayor** que el real.

### 2) RMSE (Root Mean Squared Error / Raíz del Error Cuadrático Medio)
Es un solo número que resume "en promedio, qué tan grande es el error" del modelo,
expresado en las mismas unidades que "y" (en nuestro caso, grados de temperatura).
Cuanto **más pequeño** sea el RMSE, mejor está prediciendo el modelo.

### 3) R² (Coeficiente de determinación)
Es un número entre 0 y 1 (normalmente) que indica **qué porcentaje de la variación
de los datos es explicado por el modelo**.

- R² = 1 → el modelo explica el 100% de los datos (ajuste perfecto).
- R² = 0 → el modelo no explica nada (tan malo como usar el promedio).
- Cuanto **más cerca de 1**, mejor.


In [ ]:
# --- 1) Error residual ---
# Restamos: valor real menos valor predicho, para cada punto.
df["residuo"] = df["t_medida_ruidosa"] - df["prediccion"]

print("Tabla con el error residual de cada punto:")
display(df[["t_ref", "t_medida_ruidosa", "prediccion", "residuo"]])

# --- 2) RMSE ---
# mean_squared_error calcula el "error cuadrático medio" (MSE).
mse = mean_squared_error(df["t_medida_ruidosa"], df["prediccion"])
# La raíz cuadrada del MSE nos da el RMSE, en las mismas unidades que "y".
rmse = np.sqrt(mse)

# --- 3) R² ---
# r2_score compara qué tan bien predice el modelo comparado con simplemente
# usar el promedio de "y" como predicción.
r2 = r2_score(df["t_medida_ruidosa"], df["prediccion"])

print("\n===== MÉTRICAS DEL MODELO =====")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.4f}")
print(f"R²  (Coeficiente de determinación):     {r2:.4f}")


## Paso 8 — Graficar los puntos y la recta de regresión

Por último, vamos a gráficar:

1. Los **puntos originales** (datos reales), en azul.
2. La **recta de regresión** que encontró el modelo, en rojo.

Esto nos permite ver "a simple vista" qué tan bien la recta se ajusta a los puntos.


In [ ]:

# Creamos una figura y un eje (ax) donde vamos a dibujar.
fig, ax = plt.subplots()

# 1) Dibujamos los puntos reales como un scatter plot (puntos sueltos).
ax.scatter(df["t_ref"], df["t_medida_ruidosa"],
           color="tab:blue", label="Datos medidos", zorder=3)

# 2) Dibujamos la recta de regresión.
# Usamos las predicciones del modelo, ordenadas según t_ref, para dibujar una línea continua.
orden = df["t_ref"].argsort()
ax.plot(df["t_ref"].values[orden], df["prediccion"].values[orden],
        color="tab:red", linewidth=2, label="Recta de regresión", zorder=2)

# (Opcional) Dibujamos líneas verticales punteadas que muestran el residuo de cada punto.
for _, fila in df.iterrows():
    ax.plot([fila["t_ref"], fila["t_ref"]],
            [fila["t_medida_ruidosa"], fila["prediccion"]],
            color="gray", linestyle="--", linewidth=1, zorder=1)

# Etiquetas y título del gráfico.
ax.set_xlabel("Temperatura de referencia (t_ref)")
ax.set_ylabel("Temperatura medida (t_medida)")
ax.set_title(f"Regresión Lineal — RMSE = {rmse:.3f} | R² = {r2:.3f} | nivel_ruido = {nivel_ruido}")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()


## 🎓 Para practicar

1. Vuelve a la celda del **Paso 3** y cambia `nivel_ruido` a valores como `1`, `3`, `5` o `10`.
2. Vuelve a ejecutar **todas las celdas** (por ejemplo, con el menú `Kernel → Restart & Run All`).
3. Observa cómo cambian:
   - Los puntos en el gráfico (más dispersos).
   - El valor del **RMSE** (debería subir).
   - El valor del **R²** (debería bajar, alejándose de 1).

Esto te ayudará a entender de forma intuitiva por qué el **ruido en las mediciones**
hace que sea más difícil (o menos confiable) encontrar una relación lineal perfecta.
